# Phase 1 초기 CATE 학습 노트북

**Phase 1(순수 RCT)이 끝난 후 1회만 돌리는 노트북**입니다.

### 앱별 모델 분리
앱별로 모델이 다릅니다. `APP_ID`를 바꾸면 다른 앱의 모델을 학습할 수 있어요.

Phase 1에서는 모든 유저에게 trigger를 100% 랜덤 배정했습니다.  
이 데이터(`rct_data.csv`, 8000명)로 첫 CATE 모델을 만듭니다.

이후에는 `weekly_training.ipynb`에서 매주 재학습합니다.  
(매주 데이터에서 20% 랜덤 유저를 활용)

### 이 노트북이 하는 일
1. RCT 데이터 로드 (Phase 1에서 수집)
2. ATE 분석 — trigger별 평균 클릭률 비교
3. CATE 모델 학습 (T-Learner)
4. 결과 확인 — 유저별 best trigger
5. 모델 저장 → `models/{APP_ID}/cate_model.pkl`

### 핵심 개념
- **ATE** (Average Treatment Effect): trigger 간 평균 효과 차이
- **CATE** (Conditional Average Treatment Effect): 유저별 효과 차이
- **T-Learner**: trigger별로 모델을 따로 학습하는 방법

---
## Step 1: RCT 데이터 로드

Phase 1에서 모은 데이터입니다.  
**모든 유저가 100% 랜덤으로** 4개 trigger + control 중 하나를 받았습니다.  
(Phase 2 이후의 weekly 데이터와 다르게, 여기는 모델 추천 유저가 없음)

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 앱별로 모델이 다릅니다. APP_ID를 바꾸면 다른 앱의 모델을 학습할 수 있어요.
APP_ID = "ablog"

# 앱별 모델 저장 디렉토리 생성
os.makedirs(f'../models/{APP_ID}', exist_ok=True)

# RCT 데이터 로드
rct = pd.read_csv('../data/rct_data.csv')

# app_id 컬럼이 있으면 해당 앱만 필터링
if 'app_id' in rct.columns:
    rct = rct[rct['app_id'] == APP_ID].reset_index(drop=True)

print(f'앱: {APP_ID}')
print(f'총 유저 수: {len(rct)}')
print(f'전체 모달 클릭률: {rct["modal_clicked"].mean():.1%}')
print()
print('Trigger별 배정 수:')
print(rct['assigned_trigger'].value_counts())

---
## Step 2: ATE 분석 — Trigger 간 평균 효과 차이

먼저 "trigger별로 평균적으로 효과가 있는지"를 확인합니다.  
여기서 효과가 없으면 CATE를 해도 의미가 없어요.

Phase 1은 control 그룹이 있으므로 control 대비 ATE를 계산합니다.

In [2]:
TRIGGERS = ['price_appeal', 'social_proof', 'scarcity', 'novelty']

# Control 그룹 클릭률
control = rct[rct['assigned_trigger'] == 'control']
control_rate = control['modal_clicked'].mean()
print(f'Control 클릭률: {control_rate:.1%} (n={len(control)})')
print()

# 각 Trigger vs Control 비교
print(f'{"Trigger":15s} | {"클릭률":>8s} | {"ATE":>8s} | {"p-value":>10s} | 유의미?')
print('-' * 65)

for trigger in TRIGGERS:
    treat = rct[rct['assigned_trigger'] == trigger]
    treat_rate = treat['modal_clicked'].mean()
    ate = treat_rate - control_rate
    
    _, p_val = stats.ttest_ind(treat['modal_clicked'], control['modal_clicked'])
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
    
    print(f'{trigger:15s} | {treat_rate:>7.1%} | {ate:>+7.1%} | {p_val:>10.4f} | {sig}')

print()
print('* p<0.05, ** p<0.01, *** p<0.001, n.s. = 유의미하지 않음')

Control 클릭률: 15.5% (n=1558)

Trigger         |      클릭률 |      ATE |    p-value | 유의미?
-----------------------------------------------------------------
price_appeal    |   18.2% |   +2.7% |     0.0465 | *
social_proof    |   24.7% |   +9.2% |     0.0000 | ***
scarcity        |   22.3% |   +6.7% |     0.0000 | ***
novelty         |   21.5% |   +5.9% |     0.0000 | ***

* p<0.05, ** p<0.01, *** p<0.001, n.s. = 유의미하지 않음


---
## Step 3: CATE 모델 학습 (T-Learner)

**T-Learner 방식**: trigger별로 별도 모델을 학습합니다.

```
Price trigger 모델:   Price를 받은 유저들의 (features → 클릭 여부) 학습
Social Proof 모델:    Social Proof를 받은 유저들의 (features → 클릭 여부) 학습
...
```

Phase 1 데이터는 전부 랜덤 배정이므로, 8000명 전원을 사용합니다.  
(weekly_training에서는 20% 랜덤 유저만 사용하는 것과 다름)

In [3]:
# Feature 정의
UA_FEATURES = [
    'trackinglink_count', 'DA_count', 'SA_count',
    'unique_channel_count', 'channel_entropy', 'last_touch_is_da',
    'latency', 'recency', 'recent_touch_pressure',
    'touch_per_latency_hour', 'last1h_touch_count', 'recent_24h_ratio',
    'click_ratio', 'impression_count', 'is_single_touch_install'
]

INAPP_FEATURES = [
    'product_viewed_count', 'user_signin', 'product_addedtocart',
    'deeplink_open', 'home_viewed', 'addtowishlist',
    'onboarding', 'user_signup', 'total_events', 'n_event_types'
]

CATE_FEATURES = UA_FEATURES + INAPP_FEATURES
X = rct[CATE_FEATURES].values

# Trigger별 모델 학습 (Phase 1이라 전원 사용)
ALL_TRIGGERS = TRIGGERS + ['control']
trigger_models = {}

print('Trigger별 모델 학습 (Phase 1 — 전원 RCT 데이터):')
for trigger in ALL_TRIGGERS:
    mask = rct['assigned_trigger'] == trigger
    X_t = X[mask]
    y_t = rct.loc[mask, 'modal_clicked'].values
    
    model = RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42
    )
    model.fit(X_t, y_t)
    trigger_models[trigger] = model
    
    print(f'  {trigger:15s}: n={mask.sum()}, 클릭률={y_t.mean():.1%}')

print('\n모든 모델 학습 완료!')

Trigger별 모델 학습 (Phase 1 — 전원 RCT 데이터):


  price_appeal   : n=1600, 클릭률=18.2%


  social_proof   : n=1539, 클릭률=24.7%


  scarcity       : n=1639, 클릭률=22.3%


  novelty        : n=1664, 클릭률=21.5%
  control        : n=1558, 클릭률=15.5%

모든 모델 학습 완료!


---
## Step 4: 결과 확인 — 유저별 best trigger

전체 유저에 대해 각 trigger의 클릭 확률을 예측하고, best trigger를 선택합니다.

In [4]:
# 전체 유저에 대해 trigger별 클릭 확률 예측
trigger_probs = {}
for trigger in TRIGGERS:  # control 제외
    trigger_probs[trigger] = trigger_models[trigger].predict_proba(X)[:, 1]

prob_df = pd.DataFrame(trigger_probs)

# 유저별 best trigger
rct['predicted_best'] = prob_df.idxmax(axis=1)

print('=== 유저별 Best Trigger 분포 ===')
print(rct['predicted_best'].value_counts())
print()

# 처음 5명 예시
print('=== 처음 5명의 trigger별 클릭 확률 ===')
sample = prob_df.head(5).round(3)
sample['best'] = prob_df.head(5).idxmax(axis=1)
print(sample)

=== 유저별 Best Trigger 분포 ===
predicted_best
social_proof    4150
scarcity        1931
novelty         1569
price_appeal     350
Name: count, dtype: int64

=== 처음 5명의 trigger별 클릭 확률 ===
   price_appeal  social_proof  scarcity  novelty          best
0         0.245         0.187     0.249    0.224      scarcity
1         0.188         0.227     0.198    0.255       novelty
2         0.097         0.296     0.188    0.198  social_proof
3         0.212         0.250     0.269    0.169      scarcity
4         0.128         0.250     0.248    0.220  social_proof


### 어떤 유저가 어떤 trigger를 선호하는지 분석

각 trigger를 best로 가진 유저들의 평균 프로필을 비교합니다.  
차이가 있으면 CATE 모델이 잘 작동하는 것!

In [5]:
key_features = ['recent_touch_pressure', 'channel_entropy', 
                'product_viewed_count', 'user_signin']

print('=== Trigger별 유저 프로필 (평균) ===')
print(f'{"":25s}', end='')
for trigger in TRIGGERS:
    print(f'{trigger:>15s}', end='')
print(f'{"전체":>10s}')
print('-' * 90)

for feat in key_features:
    print(f'{feat:25s}', end='')
    for trigger in TRIGGERS:
        mask = rct['predicted_best'] == trigger
        val = rct.loc[mask, feat].mean()
        print(f'{val:>15.2f}', end='')
    print(f'{rct[feat].mean():>10.2f}')

=== Trigger별 유저 프로필 (평균) ===
                            price_appeal   social_proof       scarcity        novelty        전체
------------------------------------------------------------------------------------------
recent_touch_pressure               1.29           0.50           1.00           1.00      0.75
channel_entropy                     0.75           0.81           0.65           0.74      0.75
product_viewed_count                1.71           1.34           2.33           3.40      2.00
user_signin                         0.52           0.78           0.51           0.65      0.68


---
## Step 5: 모델 저장

이 파일을 서버에 올리면 **exploration → optimized 모드로 자동 전환**됩니다.  
이후에는 `weekly_training.ipynb`에서 매주 갱신합니다.

In [ ]:
# CATE 모델 저장
cate_data = {
    'trigger_models': trigger_models,
    'treatment_triggers': TRIGGERS,
    'feature_names': CATE_FEATURES,
}

with open(f'../models/{APP_ID}/cate_model.pkl', 'wb') as f:
    pickle.dump(cate_data, f)

size = os.path.getsize(f'../models/{APP_ID}/cate_model.pkl') / 1024 / 1024
print(f'../models/{APP_ID}/cate_model.pkl 저장 완료! ({size:.1f} MB)')
print(f'  학습 데이터: Phase 1 RCT {len(rct)}명 전원 사용')
print()
print('이 파일을 서버에 업로드하면:')
print('  - 서버가 cate_model.pkl을 감지')
print('  - exploration 모드 → optimized 모드로 자동 전환')
print('  - 80% 모델 추천 + 20% 랜덤 배정 시작')

---
## Step 6: 테스트 — 유저 1명 예측

서버에서 optimized 모드일 때의 trigger score 예측을 재현합니다.

In [7]:
# 유저 1명 선택
user = rct.iloc[0]
user_features = user[CATE_FEATURES].values.astype(float).reshape(1, -1)

print(f'===== 유저 {user["user_id"]} — Trigger Score 예측 =====')
print()

scores = {}
for trigger in TRIGGERS:
    prob = trigger_models[trigger].predict_proba(user_features)[0][1]
    scores[trigger] = round(prob, 4)
    print(f'  {trigger:15s}: 클릭 확률 {prob:.4f}')

best = max(scores, key=scores.get)
print(f'\n  → best_trigger: {best}')

# API 응답 형태
print(f'\n===== API 응답 (예시) =====')
response = {
    'user_id': user['user_id'],
    'mode': 'optimized',
    'best_trigger': best,
    'trigger_scores': scores,
}
print(json.dumps(response, indent=2))

===== 유저 rct_00000 — Trigger Score 예측 =====

  price_appeal   : 클릭 확률 0.2455
  social_proof   : 클릭 확률 0.1872
  scarcity       : 클릭 확률 0.2490
  novelty        : 클릭 확률 0.2237

  → best_trigger: scarcity

===== API 응답 (예시) =====
{
  "user_id": "rct_00000",
  "mode": "optimized",
  "best_trigger": "scarcity",
  "trigger_scores": {
    "price_appeal": 0.2455,
    "social_proof": 0.1872,
    "scarcity": 0.249,
    "novelty": 0.2237
  }
}


---
## Step 7: 서버에 CATE 모델 업로드

서버에 cate_model.pkl을 업로드하면 exploration → optimized 모드로 전환됩니다.

**SERVER_URL을 실제 서버 주소로 바꿔주세요.**

In [ ]:
import requests

# 서버 주소 (로컬: http://localhost:8000, Railway: https://your-app.railway.app)
SERVER_URL = "http://localhost:8000"

filepath = f"../models/{APP_ID}/cate_model.pkl"
filename = "cate_model.pkl"

print(f"=== CATE 모델 업로드 ({APP_ID}) ===")
print(f"서버: {SERVER_URL}")
print()

try:
    with open(filepath, "rb") as f:
        resp = requests.post(
            f"{SERVER_URL}/v1/models/{APP_ID}/upload",
            files={"file": (filename, f)},
        )
    
    if resp.status_code == 200:
        result = resp.json()
        print(f"  [OK] {filename} → {result['mode']}")
        print(f"  {result['message']}")
    else:
        print(f"  [ERROR] {resp.status_code}: {resp.text}")

except requests.ConnectionError:
    print(f"  [ERROR] 서버에 연결할 수 없습니다: {SERVER_URL}")
    print(f"  → 서버를 먼저 실행하거나, SERVER_URL을 확인하세요")